In [ ]:
# =====================================================================
#  Uummannaq Sea-Ice Phenology 2017-2025  ·  Notebook cell
# =====================================================================
import pandas as pd, numpy as np
import plotly.express as px, plotly.graph_objects as go
from pathlib import Path
from scipy.stats import linregress, gaussian_kde

# ----------------------------- PARAMETERS ----------------------------
CSV_PATH       = Path("summary_test.csv")

# physical / observational constants
SUNLIT_START   = 45      # ≈ mid-Feb  → Sentinel-2 daylight returns
SUNLIT_END     = 180     # ≈ end-Jun  → melt season complete
SMOOTH_DAYS    = 5       # centred rolling mean
HEAVY_THR      = 0.60    # thick, land-fast ice
GOOD_THR       = 0.30    # “usable” ice (e.g. dog-sled routes)
LOW_THR        = 0.10    # practically open water
PERSIST_DAYS   = 14      # sustain criterion for “great decline”
# ---------------------------------------------------------------------

# ----------------------------- LOAD & PREP ---------------------------
df = pd.read_csv(CSV_PATH)
df["date"] = pd.to_datetime(df["timestamp"], format="%Y%m%dT%H%M%S")
df["year"] = df["date"].dt.year
df["doy"]  = df["date"].dt.dayofyear
df["ice_raw"] = df["solid_pct"] + df["light_pct"]
df.sort_values("date", inplace=True)

# rolling-mean smoothing in physical time (handles missing days)
df["ice"] = (
    df.groupby("year")["ice_raw"]
      .transform(lambda s: s.rolling(SMOOTH_DAYS, center=True, min_periods=1).mean())
)

# restrict to common daylight window
day_mask = df["doy"].between(SUNLIT_START, SUNLIT_END)
df_win   = df[day_mask].copy()

# ----------------------- PHENOLOGY METRICS per season ----------------
records=[]
for yr, g in df_win.groupby("year"):
    g = g.reset_index(drop=True)

    # mean ice fraction normalised by # observed days
    mean_ice = g["ice"].mean()

    # total “ice-days” (fraction·day) normalised by coverage
    ice_days = g["ice"].sum()

    # first HEAVY_THR crossing (“freeze-up” proxy)
    heavy = g[g["ice"] >= HEAVY_THR]
    freeze_doy = int(heavy["doy"].min()) if not heavy.empty else np.nan

    # sustained LOW_THR dip (“break-up”)
    decline_doy = np.nan
    for i in range(len(g) - PERSIST_DAYS):
        if g.at[i,"ice"] < LOW_THR and (g.loc[i:i+PERSIST_DAYS-1,"ice"] < LOW_THR).all():
            decline_doy = int(g.at[i,"doy"]); break

    # days with “good” ice
    good_days = int((g["ice"] >= GOOD_THR).sum())

    # steepest 7-day drop (vigour of melt)
    drop7 = g["ice"].shift(-7) - g["ice"]
    steep_drop = float((-drop7).max())

    records.append(dict(year=yr, mean_ice=mean_ice, ice_days=ice_days,
                        freeze_doy=freeze_doy, decline_doy=decline_doy,
                        good_days=good_days, steep_drop=steep_drop))

met = pd.DataFrame(records).dropna(subset=["mean_ice"])\
                           .sort_values("year")
# quick helpers
def add_trend(fig, x, y):
    slope, intercept, *_ = linregress(x, y)
    fig.add_scatter(x=x, y=slope*x+intercept,
                    mode="lines", line=dict(color="black", dash="dash"),
                    showlegend=False)

# =====================  FIG 1 · Heat-map  ============================
heat = (df_win.pivot_table(index="year", columns="doy",
                           values="ice", aggfunc="mean")
          .reindex(met["year"]))                                         
fig_heat = px.imshow(
    heat, aspect="auto", origin="lower",
    color_continuous_scale="ice",
    labels=dict(color="Ice fraction"),
    title=f"Sea-ice fraction heat-map · DOY {SUNLIT_START}-{SUNLIT_END}"
)
fig_heat.update_xaxes(tick0=SUNLIT_START, dtick=20, title="Day of year")
fig_heat.update_yaxes(title="Year", autorange="reversed")
fig_heat.show()

# =====================  FIG 2 · Mean ice bar  ========================
fig_bar = px.bar(
    met, x="year", y="mean_ice", text_auto=".2f",
    title=f"Average ice fraction ({SUNLIT_START}-{SUNLIT_END})",
    labels={"mean_ice":"Mean fraction"},
    color="mean_ice", color_continuous_scale="ice_r"
)
fig_bar.show()

# =====================  FIG 3 · Freeze & break-up trends =============
fig_freeze = px.scatter(
    met, x="year", y="freeze_doy",
    title="Freeze-up day-of-year (first ≥60 %)",
    labels={"freeze_doy":"DOY"}, trendline="ols"
); fig_freeze.update_traces(marker_size=9); fig_freeze.show()

fig_decl = px.scatter(
    met, x="year", y="decline_doy",
    title=f"Break-up day-of-year (below 10 % for {PERSIST_DAYS} d)",
    labels={"decline_doy":"DOY"}, trendline="ols"
); fig_decl.update_traces(marker_size=9); fig_decl.show()

# =====================  FIG 4 · Season length arrows ================
fig_len = go.Figure()
for _, r in met.iterrows():
    if np.isnan(r["freeze_doy"]) or np.isnan(r["decline_doy"]): continue
    fig_len.add_shape(type="line",
        x0=r["freeze_doy"], x1=r["decline_doy"],
        y0=r["year"], y1=r["year"], line=dict(width=4))
fig_len.update_yaxes(type="category", tickvals=met["year"],
                     ticktext=met["year"])
fig_len.update_xaxes(title="Day of year")
fig_len.update_layout(
    title="Season length: freeze-up → sustained break-up"
)
fig_len.show()

# =====================  FIG 5 · Steepest 7-d drop ===================
fig_drop = px.bar(
    met, x="year", y="steep_drop",
    title="Largest seven-day melt each season",
    labels={"steep_drop":"Fraction drop in 7 d"},
    color="steep_drop", color_continuous_scale="ice_r"
)
add_trend(fig_drop, met["year"], met["steep_drop"]); fig_drop.show()

# =====================  FIG 6 · Ridgeline density ===================
dens=[]
for i, (yr, g) in enumerate(df_win.groupby("year")):
    # weight KDE by ice fraction so “thick” days contribute more
    kde = gaussian_kde(g["doy"], weights=g["ice"])
    xs = np.linspace(SUNLIT_START, SUNLIT_END, 250)
    ys = kde(xs); ys = ys/ys.max()*0.9 + i   # normalise & vertically offset
    dens.append((yr, xs, ys))

fig_ridge = go.Figure()
for yr, xs, ys in dens:
    fig_ridge.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=str(int(yr))))
fig_ridge.update_yaxes(
    tickvals=[i for i,_ in enumerate(dens)],
    ticktext=[str(int(yr)) for yr,_,_ in dens],
    title="Year"
)
fig_ridge.update_xaxes(title="Day of year")
fig_ridge.update_layout(title="Ridgeline: weighted DOY distribution (sun-lit window)")
fig_ridge.show()

# =====================  FIG 7 · Cumulative melt (latest 3 y) =========
latest_years = met["year"].tail(3).astype(int).tolist()
cum=[]
for yr in latest_years:
    g = df_win[df_win["year"]==yr].copy()
    g = g.sort_values("doy")
    total = g["ice"].sum()
    g["cum_melt"] = 1 - g["ice"].cumsum()/total
    cum.append(g[["doy","cum_melt","year"]])
cum = pd.concat(cum)

fig_cum = px.line(
    cum, x="doy", y="cum_melt", color="year",
    title=f"Cumulative melt progress (latest {len(latest_years)} seasons)",
    labels={"cum_melt":"Fraction melted"}
)
fig_cum.update_yaxes(range=[0,1])
fig_cum.show()


In [4]:
# ================================================================
#   Uummannaq sea-ice phenology — enhanced visual set
# ================================================================
import pandas as pd, numpy as np
import plotly.express as px, plotly.graph_objects as go
from pathlib import Path
from scipy.stats import gaussian_kde, linregress
from datetime import datetime, timedelta

# --------------------------- PARAMETERS -------------------------
CSV = Path("summary_test.csv")

SUN_START, SUN_END = 45, 180       # daylight window DOY
SMOOTH = 5                         # days
FREEZE_THR = 0.40                  # gentler than 0.60
BREAK_THR  = 0.15
PERSIST_D  = 10                    # sustain for break
GOOD_THR   = 0.30                  # “usable” ice
CUM_YEARS  = 5                     # how many full seasons in cum plot
EARLY_EPOCH, LATE_EPOCH = (2017, 2020), (2021, 2024)
# ----------------------------------------------------------------

# --------------------------- LOAD & PREP ------------------------
df = pd.read_csv(CSV)
df["date"] = pd.to_datetime(df["timestamp"], format="%Y%m%dT%H%M%S")
df["year"] = df["date"].dt.year
df["doy"]  = df["date"].dt.dayofyear
df["ice_raw"] = df["solid_pct"] + df["light_pct"]
df = df.sort_values("date")

# rolling mean
df["ice"] = (df.groupby("year")["ice_raw"]
               .transform(lambda s: s.rolling(SMOOTH, center=True,
                                              min_periods=1).mean()))

# daylight window
df_win = df[df["doy"].between(SUN_START, SUN_END)].copy()

# ------------------ SEASON METRICS --------------------------------
records=[]
for yr, g in df_win.groupby("year"):
    g = g.reset_index(drop=True)

    # first FREEZE_THR crossing
    freeze = g[g["ice"] >= FREEZE_THR]
    freeze_doy = int(freeze["doy"].min()) if not freeze.empty else np.nan

    # sustained low ice
    decline_doy = np.nan
    for i in range(len(g)-PERSIST_D):
        if g.at[i,"ice"] < BREAK_THR and (g.loc[i:i+PERSIST_D-1,"ice"]<BREAK_THR).all():
            decline_doy = int(g.at[i,"doy"]); break

    good_days  = int((g["ice"] >= GOOD_THR).sum())
    mean_ice   = g["ice"].mean()
    ice_area   = g["ice"].sum()
    drop7      = (-(g["ice"].shift(-7) - g["ice"])).max()

    records.append(dict(year=yr, freeze_doy=freeze_doy,
                        decline_doy=decline_doy, good_days=good_days,
                        mean_ice=mean_ice, ice_area=ice_area,
                        steep_drop=drop7))

met = pd.DataFrame(records).sort_values("year")

# helper: DOY→date string for ticks
def doy_to_date(doy): return (datetime(2021,1,1)+timedelta(doy-1)).strftime("%b")

month_ticks = [45, 60, 90, 120, 150, 180]
month_lbls  = [doy_to_date(d) for d in month_ticks]

# ======================== FIG 1 · Season arrows ==================
fig_len = go.Figure()
for _, r in met.iterrows():
    if np.isnan(r["freeze_doy"]) or np.isnan(r["decline_doy"]): continue
    fig_len.add_shape(type="line",
        x0=r["freeze_doy"], x1=r["decline_doy"],
        y0=r["year"], y1=r["year"],
        line=dict(width=4))
fig_len.update_yaxes(type="category", tickvals=met["year"],
                     ticktext=[str(int(y)) for y in met["year"]])
fig_len.update_xaxes(title="Month / Day-of-Year",
                     tickvals=month_ticks, ticktext=month_lbls)
fig_len.update_layout(
    title="Season length: freeze-up (≥40 %) → break-up (≤15 %)",
    height=400)
fig_len.show()

# ======================== FIG 2 · Mean ice bars ==================
fig_bar = px.bar(
    met, x="year", y="mean_ice", text_auto=".2f",
    title=f"Average spring ice (DOY {SUN_START}-{SUN_END})",
    labels={"mean_ice":"Mean fraction"}, color="mean_ice",
    color_continuous_scale="ice_r"
)
fig_bar.show()

# ======================== FIG 3 · Heat-map =======================
heat = (df_win.pivot_table(index="year", columns="doy",
                           values="ice", aggfunc="mean")
          .reindex(met["year"]))
fig_heat = px.imshow(
    heat, aspect="auto", origin="lower", color_continuous_scale="ice",
    labels=dict(color="Ice fraction"),
    title="Sea-ice heat-map (grey rows = missing data)")
fig_heat.update_xaxes(tickvals=month_ticks, ticktext=month_lbls)
fig_heat.update_yaxes(title="Year", autorange="reversed")
fig_heat.show()

# ======================== FIG 4 · Bundled epochs + 2025 ==========
def epoch(dfsub):
    return (dfsub.groupby("doy")["ice"]
            .agg(mean="mean",
                 q25=lambda s:s.quantile(0.25),
                 q75=lambda s:s.quantile(0.75))
            .reset_index())

early = epoch(df_win[df_win["year"].between(*EARLY_EPOCH)])
late  = epoch(df_win[df_win["year"].between(*LATE_EPOCH)])
fig_ep = go.Figure()
# early band
fig_ep.add_trace(go.Scatter(x=early["doy"], y=early["q75"],
                            line=dict(width=0), showlegend=False))
fig_ep.add_trace(go.Scatter(x=early["doy"], y=early["q25"],
                            fill='tonexty', fillcolor="rgba(30,136,229,0.3)",
                            line=dict(width=0), name=f"{EARLY_EPOCH[0]}-{EARLY_EPOCH[1]} IQR"))
fig_ep.add_trace(go.Scatter(x=early["doy"], y=early["mean"],
                            line=dict(width=3, color="royalblue"),
                            name=f"{EARLY_EPOCH[0]}-{EARLY_EPOCH[1]} mean"))
# late band
fig_ep.add_trace(go.Scatter(x=late["doy"], y=late["q75"],
                            line=dict(width=0), showlegend=False))
fig_ep.add_trace(go.Scatter(x=late["doy"], y=late["q25"],
                            fill='tonexty', fillcolor="rgba(220,20,60,0.3)",
                            line=dict(width=0), name=f"{LATE_EPOCH[0]}-{LATE_EPOCH[1]} IQR"))
fig_ep.add_trace(go.Scatter(x=late["doy"], y=late["mean"],
                            line=dict(width=3, color="firebrick"),
                            name=f"{LATE_EPOCH[0]}-{LATE_EPOCH[1]} mean"))
# 2025 line if present
if 2025 in df_win["year"].unique():
    g25 = df_win[df_win["year"]==2025]
    fig_ep.add_trace(go.Scatter(x=g25["doy"], y=g25["ice"],
                                line=dict(width=3, color="black", dash="dash"),
                                name="2025 YTD"))
fig_ep.update_xaxes(tickvals=month_ticks, ticktext=month_lbls,
                    title="Month / Day-of-Year")
fig_ep.update_yaxes(title="Ice fraction")
fig_ep.update_layout(title="Early vs late epochs ±IQR  (+ 2025 YTD)")
fig_ep.show()

# ======================== FIG 5 · Cumulative melt (5 yrs + 2025) ==
full_years = met.loc[met["decline_doy"].notna(),"year"].tail(CUM_YEARS).astype(int).tolist()
if 2025 not in full_years and 2025 in df_win["year"].unique():
    full_years.append(2025)

cum=[]
for yr in full_years:
    g = df_win[df_win["year"]==yr].copy().sort_values("doy")
    total = g["ice"].sum()
    g["cum_melt"] = 1 - g["ice"].cumsum()/total
    cum.append(g[["doy","cum_melt","year"]])
cum = pd.concat(cum)

fig_cum = px.line(
    cum, x="doy", y="cum_melt", color="year",
    line_dash="year", title="Cumulative melt progression",
    labels={"cum_melt":"Fraction melted"}
)
fig_cum.update_xaxes(tickvals=month_ticks, ticktext=month_lbls,
                     title="Month / Day-of-Year")
fig_cum.update_yaxes(range=[0,1])
fig_cum.show()


In [6]:
# ====================================================================
#  Uummannaq sea-ice 2017-2025  –  full, self-contained analysis cell
# ====================================================================
import pandas as pd, numpy as np
import plotly.express as px, plotly.graph_objects as go
from pathlib import Path
from scipy.stats import linregress, gaussian_kde
from datetime import datetime, timedelta

# ------------------------ CONSTANTS ---------------------------------
CSV            = Path("summary_test.csv")
SUN_START      = 45               # daylight window (DOY)
SUN_END        = 180
SMOOTH         = 5
FREEZE_THR     = 0.40             # ≥ 40 % = land-fast build-up
BREAK_THR      = 0.15             # ≤ 15 % sustained = open
PERSIST_D      = 10
GOOD_THR       = 0.30
# --------------------------------------------------------------------

# ------------------------ LOAD & PREP -------------------------------
df = pd.read_csv(CSV)
df["date"] = pd.to_datetime(df["timestamp"], format="%Y%m%dT%H%M%S")
df["year"] = df["date"].dt.year
df["doy"]  = df["date"].dt.dayofyear
df["ice_raw"] = df["solid_pct"] + df["light_pct"]
df.sort_values("date", inplace=True)

# centred rolling mean (handles data gaps)
df["ice"] = (df.groupby("year")["ice_raw"]
               .transform(lambda s: s.rolling(SMOOTH, center=True, min_periods=1).mean()))

# daylight window
dfw = df[df["doy"].between(SUN_START, SUN_END)].copy()

# helper for month labels on DOY axis
def doy_to_mon(doy): return (datetime(2021,1,1)+timedelta(doy-1)).strftime("%b")
month_ticks = [45, 60, 91, 121, 152, 180]   # mid-month-ish
month_lbls  = [doy_to_mon(d) for d in month_ticks]

# ------------------------ METRICS per SEASON ------------------------
rec=[]
for yr, g in dfw.groupby("year"):
    g = g.sort_values("doy").reset_index(drop=True)

    # freeze-up: first record ≥ FREEZE_THR, else first DOY in window
    heavy = g[g["ice"] >= FREEZE_THR]
    freeze_doy = int(heavy["doy"].min()) if not heavy.empty else int(g["doy"].iloc[0])

    # break-up: first DOY where ice ≤ BREAK_THR for ≥ PERSIST_D
    decline_doy = np.nan
    for i in range(len(g)-PERSIST_D):
        if g.at[i,"ice"] <= BREAK_THR and (g.loc[i:i+PERSIST_D-1,"ice"]<=BREAK_THR).all():
            decline_doy = int(g.at[i,"doy"]); break

    season_len = decline_doy - freeze_doy if not np.isnan(decline_doy) else np.nan
    mean_ice   = g["ice"].mean()
    area_ice   = g["ice"].sum()
    good_days  = int((g["ice"] >= GOOD_THR).sum())

    # steepest 7-day drop
    drop7 = (-(g["ice"].shift(-7) - g["ice"])).max()

    rec.append(dict(year=yr, freeze_doy=freeze_doy, decline_doy=decline_doy,
                    season_len=season_len, mean_ice=mean_ice,
                    area_ice=area_ice, good_days=good_days,
                    steep_drop=drop7))

met = pd.DataFrame(rec).sort_values("year")

# ------------------ FIG 1  · Season-length arrows -------------------
fig_seas = go.Figure()
for _, r in met.dropna(subset=["decline_doy"]).iterrows():
    fig_seas.add_trace(go.Scatter(
        x=[r["freeze_doy"], r["decline_doy"]],
        y=[str(int(r["year"]))]*2,
        mode="lines",
        line=dict(width=6),
        showlegend=False
    ))
fig_seas.update_yaxes(title="Year", type="category")
fig_seas.update_xaxes(title="Day / Month",
                      tickvals=month_ticks, ticktext=month_lbls)
fig_seas.update_layout(title="Freeze-up → break-up timeline")
fig_seas.show()

# ------------------ FIG 2  · Average spring ice + trend -------------
fig_bar = px.bar(
    met, x="year", y="mean_ice", text_auto=".2f",
    title=f"Mean ice fraction (DOY {SUN_START}-{SUN_END})",
    labels={"mean_ice":"Mean fraction"}, color="mean_ice",
    color_continuous_scale="ice_r"
)
slope, intercept, *_ = linregress(met["year"], met["mean_ice"])
fig_bar.add_scatter(
    x=met["year"], y=slope*met["year"]+intercept,
    mode="lines", line=dict(color="black", dash="dash"),
    showlegend=False
)
fig_bar.update_layout(yaxis_range=[0,1])
fig_bar.show()

# ------------------ FIG 3  · Timing components line chart -----------
fig_comp = go.Figure()
fig_comp.add_trace(go.Scatter(x=met["year"], y=met["freeze_doy"],
                              mode="lines+markers", name="Freeze-up"))
fig_comp.add_trace(go.Scatter(x=met["year"], y=met["decline_doy"],
                              mode="lines+markers", name="Break-up"))
fig_comp.add_trace(go.Scatter(x=met["year"], y=met["season_len"],
                              mode="lines+markers", name="Season length (d)"))
fig_comp.update_yaxes(title="Day-of-year / Length",
                      tickvals=month_ticks+list(met["season_len"]),
                      ticktext=month_lbls+list(met["season_len"]))
fig_comp.update_layout(title="Phenological components over years")
fig_comp.show()

# ------------------ FIG 4  · Ridgeline KDE (window) -----------------
dens=[]
for idx, (yr, g) in enumerate(dfw.groupby("year")):
    kde = gaussian_kde(g["doy"], weights=g["ice"])
    xs = np.linspace(SUN_START, SUN_END, 250)
    ys = kde(xs)
    ys = ys / ys.max() * 0.9 + idx
    dens.append((yr, xs, ys))

fig_ridge = go.Figure()
for yr, xs, ys in dens:
    fig_ridge.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=str(int(yr))))
fig_ridge.update_yaxes(
    tickvals=[i for i,_ in enumerate(dens)],
    ticktext=[str(int(yr)) for yr,_,_ in dens],
    title="Year")
fig_ridge.update_xaxes(title="Month", tickvals=month_ticks, ticktext=month_lbls)
fig_ridge.update_layout(title="Ridgeline KDE of ice distribution")
fig_ridge.show()

# ------------------ FIG 5  · Cumulative melt (6 yrs inc. 2025) -----
latest_complete = met.dropna(subset=["decline_doy"]).tail(5)["year"].astype(int).tolist()
if 2025 not in latest_complete and 2025 in dfw["year"].unique():
    latest_complete.append(2025)

cum=[]
for yr in latest_complete:
    g = dfw[dfw["year"]==yr].copy().sort_values("doy")
    total = g["ice"].sum()
    g["cum_melt"] = 1 - g["ice"].cumsum()/total
    cum.append(g[["doy","cum_melt","year"]])
cum = pd.concat(cum)

fig_cum = px.line(
    cum, x="doy", y="cum_melt", color="year", line_dash="year",
    title="Cumulative melt progress (latest 6 seasons incl. 2025)",
    labels={"cum_melt":"Fraction melted"})
fig_cum.update_xaxes(tickvals=month_ticks, ticktext=month_lbls,
                     title="Month")
fig_cum.update_yaxes(range=[0,1])
fig_cum.show()

# ------------------ CONCLUSIONS ------------------------------------
print("Key take-aways from daylight-window analysis:")
print(" • Mean spring ice has a linear downward trend of "
      f"{slope*100:.1f} % per decade.")
recent = met.tail(6)
print(" • Break-up advanced from DOY "
      f"{int(recent['decline_doy'].iloc[0])} to "
      f"{int(recent['decline_doy'].iloc[-2])} between "
      f"{recent['year'].iloc[0]} and {recent['year'].iloc[-2]}.")
print(" • 2025 has retained only "
      f"{recent[recent['year']==2025]['mean_ice'].values[0]:.2f} mean fraction so far "
      "(data to 24 Jun).")


Key take-aways from daylight-window analysis:
 • Mean spring ice has a linear downward trend of -2.7 % per decade.
 • Break-up advanced from DOY 146 to 159 between 2020 and 2024.
 • 2025 has retained only 0.33 mean fraction so far (data to 24 Jun).


In [7]:
# ==============================================================
#  Early- vs late-epoch comparison  ·  mean ± IQR bands
#  (append directly below the main analysis cell)
# ==============================================================

EARLY_EPOCH = (2017, 2020)   # inclusive
LATE_EPOCH  = (2021, 2025)   # inclusive, includes 2025 YTD
WINDOW_STR  = f"DOY {SUN_START}-{SUN_END}"

def epoch_stats(df_sub):
    return (df_sub.groupby("doy")["ice"]
            .agg(mean="mean",
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75))
            .reset_index())

early = epoch_stats(dfw[dfw["year"].between(*EARLY_EPOCH)])
late  = epoch_stats(dfw[dfw["year"].between(*LATE_EPOCH)])

fig_epoch = go.Figure()

# early epoch
fig_epoch.add_trace(go.Scatter(
    x=early["doy"], y=early["q75"],
    line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig_epoch.add_trace(go.Scatter(
    x=early["doy"], y=early["q25"],
    fill='tonexty', fillcolor="rgba(66,135,245,0.25)",
    line=dict(width=0), name=f"{EARLY_EPOCH[0]}-{EARLY_EPOCH[1]} IQR"))
fig_epoch.add_trace(go.Scatter(
    x=early["doy"], y=early["mean"],
    line=dict(width=3, color="royalblue"),
    name=f"{EARLY_EPOCH[0]}-{EARLY_EPOCH[1]} mean"))

# late epoch
fig_epoch.add_trace(go.Scatter(
    x=late["doy"], y=late["q75"],
    line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig_epoch.add_trace(go.Scatter(
    x=late["doy"], y=late["q25"],
    fill='tonexty', fillcolor="rgba(214,41,41,0.25)",
    line=dict(width=0), name=f"{LATE_EPOCH[0]}-{LATE_EPOCH[1]} IQR"))
fig_epoch.add_trace(go.Scatter(
    x=late["doy"], y=late["mean"],
    line=dict(width=3, color="firebrick"),
    name=f"{LATE_EPOCH[0]}-{LATE_EPOCH[1]} mean"))

# axes styling
fig_epoch.update_xaxes(
    title="Month / Day-of-Year",
    tickvals=[45,60,91,121,152,180],   # Feb-Jun mid-months
    ticktext=[(datetime(2021,1,1)+timedelta(d-1)).strftime("%b") for d in
              [45,60,91,121,152,180]])
fig_epoch.update_yaxes(title="Ice fraction", range=[0,1])

fig_epoch.update_layout(
    title=f"Early vs late seasons – mean ± IQR  ({WINDOW_STR})",
    hovermode="x unified"
)

fig_epoch.show()


In [8]:
# ======================================================================
#  FINDINGS & DIFFERENCE ANALYSIS  ·  append after previous cells
# ======================================================================
from scipy.stats import linregress
import numpy as np
import plotly.graph_objects as go

# ---- 1. Epoch-mean difference curve ----------------------------------
diff_curve = late.copy()
diff_curve["diff"] = late["mean"] - early["mean"]

fig_diff = go.Figure()
fig_diff.add_trace(go.Scatter(
    x=diff_curve["doy"], y=diff_curve["diff"],
    mode="lines", line=dict(color="darkorange", width=3),
    name="Late – Early"
))
fig_diff.add_hline(y=0, line_dash="dot", line_color="grey")
fig_diff.update_xaxes(
    title="Day / Month",
    tickvals=[45,60,91,121,152,180],
    ticktext=[(datetime(2021,1,1)+timedelta(d-1)).strftime("%b")
              for d in [45,60,91,121,152,180]])
fig_diff.update_yaxes(title="Δ Ice fraction (late – early)",
                      range=[-0.6,0.6])
fig_diff.update_layout(
    title="Change in mean ice fraction profile (2021-25 minus 2017-20)",
    hovermode="x unified"
)
fig_diff.show()

# ---- 2. Epoch-level summary stats -----------------------------------
early_mean  = early["mean"].mean()
late_mean   = late ["mean"].mean()
delta_mean  = late_mean - early_mean

freeze_early = met[met["year"].between(2017,2020)]["freeze_doy"].mean()
freeze_late  = met[met["year"].between(2021,2025)]["freeze_doy"].mean()
decl_early   = met[met["year"].between(2017,2020)]["decline_doy"].mean()
decl_late    = met[met["year"].between(2021,2025)]["decline_doy"].mean()
len_early    = met[met["year"].between(2017,2020)]["season_len"].mean()
len_late     = met[met["year"].between(2021,2025)]["season_len"].mean()

# ---- 3. Linear trends (2017-25) -------------------------------------
def trend(series):
    m, b, r, p, _ = linregress(met["year"], series)
    return m, p

slope_mean,  p_mean  = trend(met["mean_ice"])
slope_frz , p_frz    = trend(met["freeze_doy"])
slope_decl, p_decl   = trend(met["decline_doy"])
slope_len , p_len    = trend(met["season_len"])

# ---- 4. Console report ----------------------------------------------
print("\n=== CLIMATE-SIGNAL FINDINGS (DOY window "
      f"{SUN_START}-{SUN_END}, smoothed {SMOOTH} d) ===")
print(f"• Mean spring ice fraction dropped by "
      f"{delta_mean:+.2f} ({late_mean:.2f} vs {early_mean:.2f}).")
print(f"• Freeze-up now occurs {freeze_late-freeze_early:+.1f} days "
      f"{'later' if freeze_late>freeze_early else 'earlier'} "
      f"(avg DOY {freeze_early:.0f} → {freeze_late:.0f}).")
print(f"• Break-up arrives {decl_late-decl_early:+.1f} days "
      f"{'later' if decl_late>decl_early else 'earlier'} "
      f"(DOY {decl_early:.0f} → {decl_late:.0f}).")
print(f"• Heavy-ice season shortened by "
      f"{len_late-len_early:+.1f} days ({len_early:.1f} → {len_late:.1f}).\n")

print("Linear trends 2017-25:")
print(f"   – Mean ice  : {slope_mean*10:+.3f}/decade  (p={p_mean:.3f})")
print(f"   – Freeze-up : {slope_frz *10:+.1f} d/decade (p={p_frz :.3f})")
print(f"   – Break-up  : {slope_decl*10:+.1f} d/decade (p={p_decl:.3f})")
print(f"   – Season len: {slope_len *10:+.1f} d/decade (p={p_len :.3f})")



=== CLIMATE-SIGNAL FINDINGS (DOY window 45-180, smoothed 5 d) ===
• Mean spring ice fraction dropped by -0.16 (0.36 vs 0.52).
• Freeze-up now occurs +3.4 days later (avg DOY 49 → 52).
• Break-up arrives -7.9 days earlier (DOY 146 → 138).
• Heavy-ice season shortened by -11.7 days (97.7 → 86.0).

Linear trends 2017-25:
   – Mean ice  : -0.268/decade  (p=0.104)
   – Freeze-up : -4.0 d/decade (p=0.695)
   – Break-up  : +nan d/decade (p=nan)
   – Season len: +nan d/decade (p=nan)


In [9]:
# ===================================================================
#  FINDINGS  ·  quantitative summary + difference curve
#  (append right after the epoch-comparison chart cell)
# ===================================================================
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from scipy.stats import linregress

# ---------- 1 · Epoch‐level aggregates --------------------------------
delta_mean   = late["mean"].mean() - early["mean"].mean()
delta_freeze = met.loc[met["year"].between(*LATE_EPOCH),"freeze_doy"].mean() - \
               met.loc[met["year"].between(*EARLY_EPOCH),"freeze_doy"].mean()
delta_break  = met.loc[met["year"].between(*LATE_EPOCH),"decline_doy"].mean() - \
               met.loc[met["year"].between(*EARLY_EPOCH),"decline_doy"].mean()
delta_len    = met.loc[met["year"].between(*LATE_EPOCH),"season_len"].mean() - \
               met.loc[met["year"].between(*EARLY_EPOCH),"season_len"].mean()
delta_drop   = met.loc[met["year"].between(*LATE_EPOCH),"steep_drop"].mean() - \
               met.loc[met["year"].between(*EARLY_EPOCH),"steep_drop"].mean()

# ---------- 2 · Linear trends 2017-2025 -------------------------------
def slope(metric):
    s,_,_,_,_ = linregress(met["year"], met[metric])
    return s
s_mean   = slope("mean_ice")
s_len    = slope("season_len")
s_break  = slope("decline_doy")
s_freeze = slope("freeze_doy")

# ---------- 3 · Console summary --------------------------------------
print("================================================================")
print("UUMMANNAQ SEA-ICE  ·  SCIENTIFIC FINDINGS (sun-lit window)")
print("----------------------------------------------------------------")
print(f"Mean spring ice dropped by {delta_mean:+.2f} (fraction) from "
      f"{EARLY_EPOCH[0]}-{EARLY_EPOCH[1]} to {LATE_EPOCH[0]}-{LATE_EPOCH[1]}.")
print(f"Freeze-up shifted {delta_freeze:+.1f} days, break-up shifted "
      f"{delta_break:+.1f} days; net season length change {delta_len:+.1f} days.")
print(f"Steepest 7-day melt increased by {delta_drop:+.2f}.")
print("--- Linear trends 2017-2025 ------------------------------------")
print(f" • Mean spring ice slope:        {s_mean*10:+.3f} / decade")
print(f" • Season length slope:          {s_len:+.2f} d / yr")
print(f" • Break-up DOY slope:           {s_break:+.2f} d / yr")
print(f" • Freeze-up DOY slope:          {s_freeze:+.2f} d / yr")
print("================================================================")

# ---------- 4 · Daily difference curve -------------------------------
diff_curve = late.merge(early, on="doy", suffixes=("_late","_early"))
diff_curve["diff"] = diff_curve["mean_late"] - diff_curve["mean_early"]

fig_diff = px.line(
    diff_curve, x="doy", y="diff",
    title="Daily ice difference  (late epoch − early epoch)",
    labels={"doy":"Day of year", "diff":"Δ ice fraction"}
)
fig_diff.add_hline(y=0, line_dash="dot")
fig_diff.update_xaxes(
    tickvals=[45,60,91,121,152,180],
    ticktext=[(datetime(2021,1,1)+timedelta(d-1)).strftime('%b')
              for d in [45,60,91,121,152,180]],
    title="Month / Day-of-Year"
)
fig_diff.show()


UUMMANNAQ SEA-ICE  ·  SCIENTIFIC FINDINGS (sun-lit window)
----------------------------------------------------------------
Mean spring ice dropped by -0.16 (fraction) from 2017-2020 to 2021-2025.
Freeze-up shifted +3.4 days, break-up shifted -7.9 days; net season length change -11.7 days.
Steepest 7-day melt increased by -0.06.
--- Linear trends 2017-2025 ------------------------------------
 • Mean spring ice slope:        -0.268 / decade
 • Season length slope:          +nan d / yr
 • Break-up DOY slope:           +nan d / yr
 • Freeze-up DOY slope:          -0.40 d / yr


In [10]:
# =====================================================================
#  FINDINGS  ·  climate-signal quantification & simple visuals
#  (append right after the main analysis cell)
# =====================================================================
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from scipy.stats import linregress
from datetime import datetime, timedelta

# --------------------- epoch definitions -----------------------------
EARLY = (2017, 2020)   # four complete seasons
LATE  = (2021, 2024)   # last complete block (exclude 2025 YTD for deltas)

# --------------------- helper: month labels --------------------------
def doy_to_mon(d): return (datetime(2021,1,1)+timedelta(d-1)).strftime("%b")
month_ticks = [45,60,91,121,152,180]
month_lbls  = [doy_to_mon(d) for d in month_ticks]

# --------------------- 1 · epoch aggregates --------------------------
def epoch_mean(metric, yr_rng):
    return met.loc[met["year"].between(*yr_rng), metric].mean()

delta_dict = {
    "Mean spring ice":   epoch_mean("mean_ice", LATE)   - epoch_mean("mean_ice", EARLY),
    "Season length (d)": epoch_mean("season_len", LATE) - epoch_mean("season_len", EARLY),
    "Freeze DOY":        epoch_mean("freeze_doy", LATE) - epoch_mean("freeze_doy", EARLY),
    "Break-up DOY":      epoch_mean("decline_doy", LATE)- epoch_mean("decline_doy", EARLY),
    "Max 7-day drop":    epoch_mean("steep_drop", LATE) - epoch_mean("steep_drop", EARLY),
}

# --------------------- 2 · linear trends 2017-2025 -------------------
def trend(metric):
    slope, intercept, r, p, _ = linregress(met["year"], met[metric])
    return slope, p
sl_mean,   p_mean   = trend("mean_ice")
sl_len,    p_len    = trend("season_len")
sl_freeze, p_freeze = trend("freeze_doy")
sl_break,  p_break  = trend("decline_doy")

# --------------------- 3 · console summary ---------------------------
print("\n================  Uummannaq daylight-window findings  ================")
print(f"Epoch {EARLY[0]}-{EARLY[1]}  vs  {LATE[0]}-{LATE[1]}:")
for k,v in delta_dict.items():
    sign = "+" if v>=0 else ""
    print(f" • {k:<18}: {sign}{v:.2f}")
print("--------------------------------------------------------------------")
print("Linear trends 2017-2025:")
print(f" • Mean spring ice       : {sl_mean*10:+.3f}/dec  (p={p_mean:.3f})")
print(f" • Season length         : {sl_len:+.2f} d/yr  (p={p_len:.3f})")
print(f" • Freeze DOY shift      : {sl_freeze:+.2f} d/yr  (p={p_freeze:.3f})")
print(f" • Break-up DOY shift    : {sl_break:+.2f} d/yr  (p={p_break:.3f})")
print("====================================================================\n")

# --------------------- 4A · delta bar-chart --------------------------
delta_df = pd.DataFrame({"Metric":delta_dict.keys(), "Delta":delta_dict.values()})
fig_delta = px.bar(
    delta_df, x="Metric", y="Delta", text="Delta",
    title=f"Change from {EARLY[0]}-{EARLY[1]} → {LATE[0]}-{LATE[1]}",
    color="Delta", color_continuous_scale="ice_r"
)
fig_delta.update_layout(xaxis_tickangle=-20)
fig_delta.show()

# --------------------- 4B · daily difference curve -------------------
# build early & late daily means on the smoothed window
def daily_curve(dfsub): return dfsub.groupby("doy")["ice"].mean().reset_index()
early_curve = daily_curve(dfw[dfw["year"].between(*EARLY)])
late_curve  = daily_curve(dfw[dfw["year"].between(*LATE)])
diff = late_curve.merge(early_curve, on="doy", suffixes=("_late","_early"))
diff["Δ"] = diff["ice_late"] - diff["ice_early"]

fig_diff = px.line(
    diff, x="doy", y="Δ",
    title=f"Late-epoch minus early-epoch ice curve  (smoothed, DOY {SUN_START}-{SUN_END})",
    labels={"Δ":"Difference in fraction"}
)
fig_diff.add_hline(y=0, line_dash="dot", line_color="grey")
fig_diff.update_xaxes(tickvals=month_ticks, ticktext=month_lbls,
                      title="Month / Day-of-Year")
fig_diff.show()



================  Uummannaq daylight-window findings  ================
Epoch 2017-2020  vs  2021-2024:
 • Mean spring ice   : -0.15
 • Season length (d) : -10.92
 • Freeze DOY        : +5.00
 • Break-up DOY      : -5.58
 • Max 7-day drop    : -0.04
--------------------------------------------------------------------
Linear trends 2017-2025:
 • Mean spring ice       : -0.268/dec  (p=0.104)
 • Season length         : +nan d/yr  (p=nan)
 • Freeze DOY shift      : -0.40 d/yr  (p=0.695)
 • Break-up DOY shift    : +nan d/yr  (p=nan)



In [12]:
# ======================================================================
#  BONUS SCIENCE CELL  ·  correlations, variability, anomaly, violins
#  (paste after previous findings cell)
# ======================================================================
import plotly.express as px, plotly.graph_objects as go
from scipy.stats import linregress
import numpy as np, pandas as pd
from datetime import datetime, timedelta

# ------------- month labels helper (reuse constants) ------------------
def doy_to_mon(d): return (datetime(2021,1,1)+timedelta(d-1)).strftime("%b")
month_ticks = [45,60,91,121,152,180]
month_lbls  = [doy_to_mon(d) for d in month_ticks]

# ---------- A · season-length vs mean ice -----------------------------
corr_df = met.dropna(subset=["season_len","mean_ice"])         # align data
fig_corr = px.scatter(
    corr_df, x="season_len", y="mean_ice", text="year",
    title="Season length vs mean spring ice",
    labels={"season_len":"Season length (d)", "mean_ice":"Mean fraction"},
)
s, b, r, p, _ = linregress(corr_df["season_len"], corr_df["mean_ice"])
fig_corr.add_scatter(
    x=corr_df["season_len"], y=s*corr_df["season_len"]+b,
    mode="lines", line=dict(color="black", dash="dash"),
    showlegend=False)
fig_corr.update_traces(marker_size=9)
fig_corr.update_layout(
    annotations=[dict(x=corr_df["season_len"].max()*0.7,
                      y=b+s*corr_df["season_len"].max()*0.7,
                      text=f"R²={r**2:.2f}, p={p:.3f}", showarrow=False)]
)
fig_corr.show()

# ---------- B · coefficient-of-variation ------------------------------
cov = (dfw.groupby("year")["ice"]
          .agg(mean="mean", sd="std")
          .assign(cov=lambda d: d["sd"]/d["mean"]))
fig_cov = px.bar(
    cov.reset_index(), x="year", y="cov",
    title="Interannual coefficient of variation (within daylight window)",
    labels={"cov":"σ / μ"}
)
fig_cov.show()

# ---------- C · anomaly vs 2017-20 baseline ----------------------------
baseline = met.loc[met["year"].between(2017,2020),"mean_ice"].mean()
met["anomaly"] = met["mean_ice"] - baseline
fig_anom = px.bar(
    met, x="year", y="anomaly",
    title="Mean spring-ice anomaly relative to 2017-2020",
    labels={"anomaly":"Δ fraction"},
    color="anomaly", color_continuous_scale="RdBu_r", range_color=[-0.4,0.4]
)
fig_anom.add_hline(y=0, line_dash="dot")
fig_anom.show()

# ---------- D · violins of freeze & break dates -----------------------
vdf = met.melt(id_vars="year",
               value_vars=["freeze_doy","decline_doy"],
               var_name="event", value_name="doy").dropna()
vdf["event"] = vdf["event"].map({"freeze_doy":"Freeze-up",
                                 "decline_doy":"Break-up"})
fig_vio = px.violin(
    vdf, x="event", y="doy", color="event",
    box=True, points=False,
    title="Distribution of phenology dates (2017-2025)",
    labels={"doy":"Day-of-year"}
)
fig_vio.update_yaxes(tickvals=month_ticks, ticktext=month_lbls)
fig_vio.show()

# ---------- console extras --------------------------------------------
worst = met.loc[met['anomaly'].idxmin()]
best  = met.loc[met['anomaly'].idxmax()]
print("--- Bonus nuggets -----------------------------------------")
print(f"Strong length–mean correlation:  R = {r:+.2f},  p = {p:.3f}")
print(f"Lowest-ice spring:  {int(worst.year)}  (Δ {worst.anomaly:+.2f})")
print(f"Highest-ice spring: {int(best.year)}   (Δ {best.anomaly:+.2f})")
print("-----------------------------------------------------------")


--- Bonus nuggets -----------------------------------------
Strong length–mean correlation:  R = +0.87,  p = 0.005
Lowest-ice spring:  2021  (Δ -0.31)
Highest-ice spring: 2018   (Δ +0.09)
-----------------------------------------------------------
